# 02 · Synthetic drawing PDFs with planted compliance flaws

Writes 1,200 mock engineering drawings (ISO, P&ID, GA, specification and symbol sheets) as PDFs into the landing Volume `/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/`, plus the ground truth the compliance judge is scored against later. Everything is synthetic and deterministic: one `seed=42` drives the layouts, the numbers and which drawings are flawed, so the same inputs always give byte-identical PDFs.

The drawing code is a normal Python module, [`02_generate_drawings.py`](./02_generate_drawings.py), unit-tested in `tests/test_generate_drawings.py`. This notebook only calls `generate_drawings(...)`, writes the two sidecar files and checks what landed in the Volume. ReportLab draws every sheet: a title block, a revision table, notes, and a schematic built from lines, boxes and text. Exactly 20% of the drawings (240) carry a planted flaw. Almost all of those carry one; about 10% carry two or three. For each flaw the generator records the bounding box of the element it got wrong.

| flaw_type | citation | severity | what is planted |
|---|---|---|---|
| `flange_rating_mismatch` | ASME B16.5 §4.2 | critical | a flange (or table row) whose class cannot hold the line's design pressure |
| `missing_pressure_relief_notation` | ASME B31.3 §322 | critical | a vessel (or schedule row) without its PSV tag and set pressure |
| `missing_loto_tag` | OSHA 1910.147 | high | a pump, compressor or powered unit drawn without its LOTO isolation tag |
| `egress_width_under_code` | IBC 1005.1 | high | an exit whose clear width is below the width its occupant load requires |
| `weld_joint_spec_mismatch` | ASME B31.3 §341 | medium | a weld (or table row) whose joint type or examination differs from the piping class |

Conventions for the tasks that consume the ground truth: `page_num` is 0-based (like `ai_parse_document` page ids), and `bbox` is `[x, y, w, h]`, each 0.0 to 1.0 of the page, origin top-left, y growing downward. The manifest (`drawing_manifest.json`) and the findings (`ground_truth_findings.jsonl`) are written next to this notebook in `outputs/`, not into the Volume, so Auto Loader sees exactly 1,200 files there.

In [1]:
%pip install -q reportlab==5.1.0 faker==40.44.0

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
import importlib
import json
import os
import platform
import sys
import time
from collections import Counter
from pathlib import Path

import faker
import numpy as np
import pandas as pd
import reportlab

# Widgets let a job override these; the defaults match the variables in databricks.yml.
dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
catalog = dbutils.widgets.get("catalog")
prefix = dbutils.widgets.get("schema_prefix")

N_DRAWINGS, FLAW_RATE, SEED = 1200, 0.2, 42
OUT_DIR = f"/Volumes/{catalog}/{prefix}_landing/drawings"
OUT = Path("outputs")  # next to this notebook: the manifest and the ground truth

# The generator sits next to this notebook. Its file name starts with a digit, so it is imported by name.
sys.path.insert(0, os.getcwd())
gen = importlib.import_module("02_generate_drawings")

print(f"python {platform.python_version()}, numpy {np.__version__}, pandas {pd.__version__}, "
      f"reportlab {reportlab.Version}, faker {faker.VERSION}")
print(f"target: {OUT_DIR}")

python 3.10.12, numpy 1.23.5, pandas 1.5.3, reportlab 5.1.0, faker 40.44.0
target: /Volumes/mfg_mc_se_sa/bluerridge_landing/drawings


## Generate the PDFs

One call writes all 1,200 files (one `write_bytes` per file, which is what Volumes support) and returns the manifest and the ground truth. The flawed drawings are chosen as an exact count, `round(1200 * 0.2) = 240`, not by a coin flip per drawing, so the 20% is not subject to sampling noise.

In [3]:
t0 = time.time()
result = gen.generate_drawings(n=N_DRAWINGS, flaw_rate=FLAW_RATE, seed=SEED, out_dir=OUT_DIR, progress_every=200)
print(f"{result.manifest_count} drawings, {result.flawed_count} with at least one planted flaw, "
      f"{result.finding_count} planted findings, {time.time() - t0:.0f} s")

200/1200 drawings written
400/1200 drawings written
600/1200 drawings written
800/1200 drawings written
1000/1200 drawings written
1200/1200 drawings written
1200 drawings, 240 with at least one planted flaw, 274 planted findings, 82 s


## Manifest and ground truth

`drawing_manifest.json` has one entry per drawing; `ground_truth_findings.jsonl` has one line per planted finding, so a drawing with three flaws appears three times (each with its own `finding_id`, page and box). These are the files committed under `01_synthetic_data/outputs/`.

In [4]:
OUT.mkdir(exist_ok=True)
result.write_manifest(OUT / "drawing_manifest.json")
result.write_ground_truth(OUT / "ground_truth_findings.jsonl")
print(f"outputs/drawing_manifest.json: {len(result.manifest)} entries")
print(f"outputs/ground_truth_findings.jsonl: {len(result.ground_truth)} lines")

print("\nManifest, first rows:")
display(spark.createDataFrame(pd.DataFrame(result.manifest)).limit(5))

print("A ground-truth entry (page_num is 0-based; bbox is [x, y, w, h], 0-1, origin top-left):")
print(json.dumps(result.ground_truth[0], indent=2, ensure_ascii=False))

per_drawing = Counter(f["drawing_id"] for f in result.ground_truth)
three = next(d for d, c in per_drawing.items() if c == 3)
print(f"\n{three} carries three planted flaws, so it has three lines:")
display(spark.createDataFrame(pd.DataFrame([f for f in result.ground_truth if f["drawing_id"] == three])))

outputs/drawing_manifest.json: 1200 entries
outputs/ground_truth_findings.jsonl: 274 lines

Manifest, first rows:


drawing_id,discipline,revision,drawing_type,volume_path,has_flaw
BRL-MEC-GA-0001,mechanical,B,GA,/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/BRL-MEC-GA-0001.pdf,False
BRL-MEC-GA-0002,mechanical,D,GA,/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/BRL-MEC-GA-0002.pdf,True
BRL-STR-GA-0003,structural,C,GA,/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/BRL-STR-GA-0003.pdf,False
BRL-PIP-ISO-0004,piping,B,ISO,/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/BRL-PIP-ISO-0004.pdf,False
BRL-STR-GA-0005,structural,A,GA,/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/BRL-STR-GA-0005.pdf,True


A ground-truth entry (page_num is 0-based; bbox is [x, y, w, h], 0-1, origin top-left):
{
  "drawing_id": "BRL-MEC-GA-0002",
  "finding_id": "BRL-MEC-GA-0002-F01",
  "page_num": 0,
  "bbox": [
    0.1625,
    0.4842,
    0.0557,
    0.076
  ],
  "citation": "OSHA 1910.147",
  "severity": "high",
  "flaw_type": "missing_loto_tag"
}

BRL-INS-PID-0052 carries three planted flaws, so it has three lines:


drawing_id,finding_id,page_num,bbox,citation,severity,flaw_type
BRL-INS-PID-0052,BRL-INS-PID-0052-F01,0,"[0.3398, 0.2222, 0.0875, 0.1104]",OSHA 1910.147,high,missing_loto_tag
BRL-INS-PID-0052,BRL-INS-PID-0052-F02,0,"[0.4948, 0.4533, 0.0975, 0.2378]",ASME B31.3 §322,critical,missing_pressure_relief_notation
BRL-INS-PID-0052,BRL-INS-PID-0052-F03,0,"[0.235, 0.5532, 0.0334, 0.0365]",ASME B16.5 §4.2,critical,flange_rating_mismatch


## Verify what landed

Read back the way a consumer would: list the Volume with `dbutils`, count it again with Spark's `binaryFile` reader (a different code path than the writer used), check each file is a PDF, and re-render a few drawings from the seed to show the bytes on the Volume are reproducible.

In [5]:
files = [f for f in dbutils.fs.ls(OUT_DIR) if f.name.endswith(".pdf")]
sizes = [f.size for f in files]
print(f"{len(files)} PDF files in {OUT_DIR} ({sum(sizes) / 1e6:.1f} MB; {min(sizes) / 1e3:.1f} to {max(sizes) / 1e3:.1f} KB each)")
assert len(files) == N_DRAWINGS
assert {f.name for f in files} == {f"{d['drawing_id']}.pdf" for d in result.manifest}

spark_count = spark.read.format("binaryFile").option("pathGlobFilter", "*.pdf").load(OUT_DIR).count()
print(f"Spark binaryFile read of the Volume: {spark_count} files")
assert spark_count == N_DRAWINGS

bad = [d["drawing_id"] for d in result.manifest if not Path(d["volume_path"]).read_bytes().startswith(b"%PDF-")]
assert not bad, bad
plans = gen.plan_drawings(N_DRAWINGS, FLAW_RATE, SEED)
for plan in (plans[i] for i in (0, 299, 599, 899, 1199)):
    pdf, _ = gen.render_drawing(plan, SEED)
    assert pdf == Path(f"{OUT_DIR}/{plan.drawing_id}.pdf").read_bytes(), plan.drawing_id
print("every file starts with %PDF-; 5 drawings re-rendered from the seed are byte-identical to their Volume copies")

manifest = pd.DataFrame(result.manifest)
findings = pd.DataFrame(result.ground_truth)
print("\ndrawing_type:", dict(Counter(manifest["drawing_type"])))
print("discipline:  ", dict(Counter(manifest["discipline"])))
print("revision:    ", dict(sorted(Counter(manifest["revision"]).items())))

per_drawing = findings.groupby("drawing_id").size()
print(f"\nflawed drawings: {result.flawed_count} of {result.manifest_count} ({result.flawed_count / result.manifest_count:.0%})")
print("findings per flawed drawing:", dict(sorted(Counter(per_drawing).items())), f"-> {result.finding_count} findings in total")
print("findings by page_num:", dict(sorted(Counter(findings["page_num"]).items())))
b = findings["bbox"].apply(pd.Series)
b.columns = ["x", "y", "w", "h"]
print(f"bbox: x {b.x.min():.3f}..{(b.x + b.w).max():.3f}, y {b.y.min():.3f}..{(b.y + b.h).max():.3f} (all inside the 0-1 page)")

assert result.manifest_count == N_DRAWINGS and result.flawed_count == round(N_DRAWINGS * FLAW_RATE) == per_drawing.size
assert set(findings["flaw_type"]) == set(gen.FLAW_TYPES)
assert per_drawing.max() == 3 and findings["finding_id"].is_unique
assert (b[["x", "y", "w", "h"]] > 0).all().all() and ((b.x + b.w) <= 1).all() and ((b.y + b.h) <= 1).all()
assert set(manifest["drawing_type"]) <= {"ISO", "PID", "GA", "SPEC", "OTHER"}

summary = (findings.groupby(["flaw_type", "citation", "severity"]).size().reset_index(name="findings")
           .sort_values("findings", ascending=False))
display(spark.createDataFrame(summary))

1200 PDF files in /Volumes/mfg_mc_se_sa/bluerridge_landing/drawings (9.1 MB; 4.6 to 15.6 KB each)
Spark binaryFile read of the Volume: 1200 files
every file starts with %PDF-; 5 drawings re-rendered from the seed are byte-identical to their Volume copies

drawing_type: {'GA': 269, 'ISO': 397, 'PID': 373, 'SPEC': 135, 'OTHER': 26}
discipline:   {'mechanical': 134, 'structural': 113, 'piping': 756, 'instr': 120, 'electrical': 77}
revision:     {'A': 124, 'B': 305, 'C': 379, 'D': 252, 'E': 140}

flawed drawings: 240 of 1200 (20%)
findings per flawed drawing: {1: 216, 2: 14, 3: 10} -> 274 findings in total
findings by page_num: {0: 255, 1: 19}
bbox: x 0.022..0.667, y 0.100..0.728 (all inside the 0-1 page)


flaw_type,citation,severity,findings
flange_rating_mismatch,ASME B16.5 §4.2,critical,63
missing_loto_tag,OSHA 1910.147,high,59
missing_pressure_relief_notation,ASME B31.3 §322,critical,58
weld_joint_spec_mismatch,ASME B31.3 §341,medium,49
egress_width_under_code,IBC 1005.1,high,45
